In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/game_logs.csv", dtype={"GAME_ID": str})
df["GAME_DATE"] = pd.to_datetime(df["GAME_DATE"])
df["WIN"] = (df["WL"] == "W").astype(int)
df["PTS_ALLOWED"] = df["PTS"] - df["PLUS_MINUS"]
df = df.sort_values(["TEAM_ID", "GAME_DATE"]).reset_index(drop=True)

In [2]:
g = df.groupby("TEAM_ID")

for col in ["PTS", "PTS_ALLOWED", "WIN"]:
    df[f"{col}_ROLL10"] = g[col].transform(
        lambda s: s.shift(1).rolling(10, min_periods=5).mean()
    )

In [3]:
df["REST_DAYS"] = g["GAME_DATE"].diff().dt.days
df["REST_DAYS"] = df["REST_DAYS"].clip(upper=7)

In [4]:
feat_cols = ["PTS_ROLL10", "PTS_ALLOWED_ROLL10", "WIN_ROLL10", "REST_DAYS"]

games = pd.read_csv("../data/processed/games.csv", dtype={"GAME_ID": str}, parse_dates=["GAME_DATE"])

home_f = df[["GAME_ID", "TEAM_ID"] + feat_cols].rename(
    columns={c: f"HOME_{c}" for c in feat_cols} | {"TEAM_ID": "HOME_TEAM_ID"}
)
away_f = df[["GAME_ID", "TEAM_ID"] + feat_cols].rename(
    columns={c: f"AWAY_{c}" for c in feat_cols} | {"TEAM_ID": "AWAY_TEAM_ID"}
)

games = games.merge(home_f, on=["GAME_ID", "HOME_TEAM_ID"], validate="one_to_one")
games = games.merge(away_f, on=["GAME_ID", "AWAY_TEAM_ID"], validate="one_to_one")

In [5]:
print("Before:", len(games))
feature_cols_all = [f"{side}_{c}" for side in ["HOME", "AWAY"] for c in feat_cols]
games = games.dropna(subset=feature_cols_all).reset_index(drop=True)
print("After:", len(games))

games.to_csv("../data/processed/features.csv", index=False)
games[["GAME_DATE", "HOME_TEAM", "AWAY_TEAM", "HOME_WIN"] + feature_cols_all].head()

Before: 11974
After: 11896


,GAME_DATE,HOME_TEAM,AWAY_TEAM,HOME_WIN,HOME_PTS_ROLL10,HOME_PTS_ALLOWED_ROLL10,HOME_WIN_ROLL10,HOME_REST_DAYS,AWAY_PTS_ROLL10,AWAY_PTS_ALLOWED_ROLL10,AWAY_WIN_ROLL10,AWAY_REST_DAYS
0,2015-11-05,CHI,OKC,1,100.6,102.0,0.6,2.0,114.2,109.6,0.6,1.0
1,2015-11-05,POR,MEM,1,101.6,99.6,0.6,1.0,92.2,101.6,0.6,2.0
2,2015-11-06,SAC,HOU,0,106.2,112.0,0.2,2.0,99.0,109.0,0.4,2.0
3,2015-11-06,GSW,DEN,1,117.6,96.8,1.0,2.0,96.0,100.4,0.4,1.0
4,2015-11-06,NYK,MIL,0,102.0,101.8,0.4,2.0,98.2,105.8,0.4,2.0


In [8]:
check = df[df["TEAM_ABBREVIATION"] == "POR"].head(11)[["GAME_DATE", "PTS", "PTS_ROLL10"]]
check

,GAME_DATE,PTS,PTS_ROLL10
15969,2015-10-28,112,NaN
15970,2015-10-30,92,NaN
15971,2015-10-31,90,NaN
15972,2015-11-02,106,NaN
15973,2015-11-04,108,NaN
15974,2015-11-05,115,101.600000
15975,2015-11-08,103,103.833333
15976,2015-11-09,104,103.714286
15977,2015-11-11,101,103.750000
15978,2015-11-13,100,103.444444
